# Tecan Spark
This notebook demonstrates how to use wetlabtools to parse data from a result file from the Tecan Spark plate reader. The results should be saved as Excel (.xlsx) files. When creating an experiment from a result file, all data parsing and formatting should happen automatically. In principle the result file contains all information to parse the data correctly.

In [1]:
import wetlabtools

result_file = "example_data/spark/absorbance_1.xlsx"
experiment = wetlabtools.spark.Experiment(result_file=result_file)
experiment

Loading BokehJS ...

Experiment(file=example_data/spark/absorbance_1.xlsx, date=2024-06-28)

The experiment class contains a variety of meta data, most of it will be completely irrelevant

In [2]:
print(experiment.user)
print(experiment.application)
print(experiment.device)

INTRANET\goldbach
SparkControl V3.2
Spark XXX


## Workflow
The workflow is a representation of the measurements and actions in the file. It manages the sequence of actions executed duriong the assay on the plate reader. The workflow can be printed as a tree representation to quickly check if the assay steps were parsed correctly.

In [3]:
workflow = experiment.workflow
print(workflow.to_tree())

Workflow(root_actions=1)
    - PlateAction, NUN96ft
        - AbsorbanceAction, A450


The workflow can also be used to find specific actions (measurements). For example, we can find all actions of type ```AbsorbanceAction```

In [4]:
from wetlabtools.spark.actions import AbsorbanceAction

workflow.find_all(AbsorbanceAction)

[AbsorbanceAction(label=A450, children=0)]

It is alos possible to iterate through all actions in the order of their execution. Note that the indent level information is lost

In [5]:
for action in workflow.iter_execution_order():
    print(action)

PlateAction, NUN96ft
AbsorbanceAction, A450


## Actions
Actions are classes to parse data from measurements and store all meta data of the respective action (e.g. settings). 

In [6]:
# find_all() returns a list
absorbance_measurement = workflow.find_all(AbsorbanceAction)[0]
absorbance_measurement.label

'A450'

In [7]:
absorbance_measurement.region

PlateRegion('A1-H6', selected_wells=48, total_wells=96, shape=8x12)

## Absorbance Measuerments
Single wavelength absorbance measurements are simply stored as a data frame containing the absorbance value for each well. 

In [8]:
result_file = 'example_data/spark/absorbance_1.xlsx'
experiment = wetlabtools.spark.Experiment(result_file=result_file)
absorbance = experiment.workflow.find_all(AbsorbanceAction)[0]
print("Wavelength Scan:", absorbance.scan)
print("Reference wavelength:", absorbance.reference)
print("Absorbance at", absorbance.wavelength, "nm:")
absorbance.data

Wavelength Scan: False
Reference wavelength: None
Absorbance at 450 nm:


,Well,value
0,A1,2.6103
1,A2,2.6000
2,A3,2.6277
3,A4,2.6083
4,A5,2.6048
...,...,...
91,H8,NaN
92,H9,NaN
93,H10,NaN
94,H11,NaN


Absorbance measurements can also be done with a reference wavelength for baseline correction. The data exported from the instrument should already contain the reference corrected data. The data is stored as ```value``` (raw absorbance measurement), ```reference``` (the absorbance value at the reference wavelength), and ```difference``` (the reference corrected value)

In [9]:
result_file = 'example_data/spark/absorbance_2.xlsx'
experiment = wetlabtools.spark.Experiment(result_file=result_file)
absorbance = experiment.workflow.find_all(AbsorbanceAction)[0]
print("Wavelength Scan:", absorbance.scan)
print("Reference wavelength:", absorbance.reference, 'nm')
print("Absorbance at", absorbance.wavelength, "nm:")
absorbance.data

Wavelength Scan: False
Reference wavelength: 570 nm
Absorbance at 450 nm:


,Well,value,reference,difference
0,A1,0.2610,0.0427,0.2183
1,A2,0.3001,0.0422,0.2579
2,A3,0.2570,0.0417,0.2153
3,A4,0.3716,0.0435,0.3281
4,A5,1.0871,0.0443,1.0428
...,...,...,...,...
91,H8,1.5086,0.0483,1.4603
92,H9,1.2825,0.0437,1.2388
93,H10,0.5234,0.0424,0.4810
94,H11,1.4188,0.0438,1.3750


If the absorbance across multiple wavelengths was measured, the respective measurement is a wavelength scan. The data contains the absorbance ```Value``` at each ```Wavelength``` for every ```Well``` recorded.

In [13]:
result_file = 'example_data/spark/absorbance_3.xlsx'
experiment = wetlabtools.spark.Experiment(result_file=result_file)
absorbance = experiment.workflow.find_all(AbsorbanceAction)[0]
print("Wavelength Scan:", absorbance.scan)
print("Scanning from", absorbance.start, "nm to", absorbance.end, "nm at", absorbance.step, "nm increments")
absorbance.data

Wavelength Scan: True
Scanning from 240 nm to 700 nm at 1 nm increments


,Wavelength,Well,Value
0,240.0,A1,3.7717
1,241.0,A1,3.8696
2,242.0,A1,3.7064
3,243.0,A1,NaN
4,244.0,A1,3.8348
...,...,...,...
4605,696.0,H1,0.0403
4606,697.0,H1,0.0402
4607,698.0,H1,0.0407
4608,699.0,H1,0.0406
